# Group By e Funções Agregadas com Pandas

## Contexto: análise de transações bancárias

O Banco Meridiano processa milhares de transações por dia entre clientes pessoa física. O time de dados recebeu a tarefa de responder perguntas operacionais e estratégicas: qual tipo de transação movimenta mais volume financeiro? Clientes de contas salário fazem PIX de valor maior ou menor que os de conta corrente? Quais estados têm o maior número de clientes ativos?

Essas perguntas não se respondem linha a linha — elas exigem **agrupar e sumarizar** os dados. É exatamente o que `groupby` e as funções agregadas fazem.

&nbsp;

### Dataset utilizado neste material

Arquivo: `transacoes_bancarias.csv` — 800 registros de transações do Banco Meridiano entre janeiro de 2023 e junho de 2024.

| Coluna             | Tipo    | Descrição                                                      |
|--------------------|---------|----------------------------------------------------------------|
| `transaction_id`   | int     | Identificador único da transação                               |
| `client_id`        | int     | Identificador do cliente                                       |
| `age_group`        | string  | Faixa etária: `18-25`, `26-35`, `36-45`, `46-55`, `56+`       |
| `state`            | string  | Estado da agência ou endereço do cliente                       |
| `account_type`     | string  | Tipo de conta: `corrente`, `poupança`, `salário`               |
| `transaction_type` | string  | Tipo: `débito`, `crédito`, `transferência`, `PIX`              |
| `category`         | string  | Categoria da transação (ex: `supermercado`, `salário`)         |
| `amount`           | float   | Valor da transação em R$ — contém nulos                        |
| `balance_after`    | float   | Saldo do cliente após a transação                              |
| `date`             | string  | Data da transação (YYYY-MM-DD)                                 |
| `status`           | string  | Status: `aprovada`, `recusada`, `estornada`                    |

&nbsp;

In [1]:
import pandas as pd
df = pd.read_csv('transacoes_bancarias.csv')
df

,transaction_id,client_id,age_group,state,account_type,transaction_type,category,amount,balance_after,date,status
0,100100,5284,36-45,RJ,corrente,débito,assinatura,276.23,1331.36,2023-01-02,recusada
1,100699,5206,18-25,BA,poupança,crédito,freelance,4811.58,7800.85,2023-01-03,aprovada
2,100413,5228,26-35,SP,corrente,PIX,supermercado,127.20,2136.24,2023-01-04,aprovada
3,100480,5139,56+,RJ,salário,crédito,freelance,11337.06,12893.01,2023-01-04,aprovada
4,100640,5028,36-45,SP,corrente,transferência,escola,1018.25,3663.90,2023-01-05,aprovada
...,...,...,...,...,...,...,...,...,...,...,...
795,100744,5085,56+,SP,corrente,PIX,farmácia,1481.92,-549.29,2024-06-28,aprovada
796,100543,5299,26-35,SP,corrente,crédito,estorno,9973.14,16041.70,2024-06-29,aprovada
797,100032,5230,46-55,RJ,poupança,transferência,aluguel,1402.02,-877.82,2024-06-29,aprovada
798,100019,5100,56+,RJ,corrente,crédito,salário,2092.14,3620.85,2024-06-30,aprovada


## Group By

`groupby` divide o DataFrame em grupos com base nos valores de uma ou mais colunas, aplica uma função sobre cada grupo e combina os resultados em uma única estrutura.

O fluxo é sempre: **Split → Apply → Combine**.

In [ ]:
df.groupby('coluna_de_agrupamento')['coluna_de_valor'].funcao_agregada()

### Agrupamento por uma coluna

In [2]:
df.groupby('transaction_type')['amount'].sum()
# # Arredondar os valores
# df.groupby('transaction_type')['amount'].sum().round(2)

transaction_type
PIX               237380.45
crédito          1067233.84
débito            178420.38
transferência     407410.63
Name: amount, dtype: float64

In [3]:
df.groupby('transaction_type')['amount'].sum().sort_values(ascending=False)

transaction_type
crédito          1067233.84
transferência     407410.63
PIX               237380.45
débito            178420.38
Name: amount, dtype: float64

In [4]:
df.groupby('state')['client_id'].nunique().sort_values(ascending=False)

state
SP    164
RJ    112
MG    107
PR     66
RS     66
BA     65
GO     47
Name: client_id, dtype: int64

### Aplicando `.agg()` com múltiplas funções sobre uma coluna

`.agg()` permite calcular várias métricas em uma única chamada, retornando um DataFrame:

In [5]:
df.groupby('transaction_type')['amount'].agg(
    total='sum',
    media='mean',
    desvio_padrao='std',
    minimo='min',
    maximo='max',
    contagem='count'
).round(2)

,total,media,desvio_padrao,minimo,maximo,contagem
transaction_type,,,,,,
PIX,237380.45,1304.29,696.17,49.96,2496.95,182
crédito,1067233.84,6314.99,3325.32,821.78,11917.65,169
débito,178420.38,626.04,343.73,24.66,1193.57,285
transferência,407410.63,2578.55,1383.86,242.08,4995.52,158


a sintaxe `nome_coluna='funcao de agregação'` nomeia a coluna diretamente, não precisando renomear depois

## Group By com Múltiplas Colunas

É possível agrupar por mais de uma coluna passando uma lista. O resultado tem um índice hierárquico (MultiIndex).

### 3.1 Agrupamento por duas colunas

In [6]:
df.groupby(['account_type','transaction_type']).agg(
    total_amount=('amount', 'sum'),
    media_amount=('amount', 'mean'),
    num_transacoes=('transaction_id', 'count')
).round(2)

total_amount  media_amount  num_transacoes
account_type transaction_type                                            
corrente     PIX                  128829.01       1275.53             103
             crédito              602905.53       6482.86              93
             débito               106016.27        623.63             171
             transferência        239143.29       2627.95              91
poupança     PIX                   70555.48       1306.58              55
             crédito              271241.99       5771.11              47
             débito                51007.43        653.94              79
             transferência        111030.09       2362.34              47
salário      PIX                   37995.96       1407.26              27
             crédito              193086.32       6658.15              30
             débito                21396.68        578.29              37
             transferência         57237.25       2861.86              20

### Diferentes funções por coluna com `.agg()`

É possível aplicar funções diferentes para cada coluna em um único `agg`:


In [7]:
df.groupby('transaction_type').agg(
    total_amount=('amount', 'sum'),
    media_amount=('amount', 'mean'),
    num_transacoes=('transaction_id', 'count'),
    clientes_distintos=('client_id', 'nunique'),
    primeira_transacao=('date', 'min'),
    ultima_transacao=('date', 'max')
).round(2)

,total_amount,media_amount,num_transacoes,clientes_distintos,primeira_transacao,ultima_transacao
transaction_type,,,,,,
PIX,237380.45,1304.29,185,136,2023-01-04,2024-06-28
crédito,1067233.84,6314.99,170,134,2023-01-03,2024-06-30
débito,178420.38,626.04,287,182,2023-01-02,2024-06-28
transferência,407410.63,2578.55,158,125,2023-01-05,2024-06-29


### Transformando o resultado com `.unstack()`

Com dois níveis de agrupamento, `.unstack()` pivota o segundo nível de índice para colunas — mais legível para comparações:

In [8]:
df.groupby(['account_type', 'transaction_type'])['amount'].mean().round(2)

account_type  transaction_type
corrente      PIX                 1275.53
              crédito             6482.86
              débito               623.63
              transferência       2627.95
poupança      PIX                 1306.58
              crédito             5771.11
              débito               653.94
              transferência       2362.34
salário       PIX                 1407.26
              crédito             6658.15
              débito               578.29
              transferência       2861.86
Name: amount, dtype: float64

In [9]:
df.groupby(['account_type', 'transaction_type'])['amount'].mean().round(2).unstack()

transaction_type,PIX,crédito,débito,transferência
account_type,,,,
corrente,1275.53,6482.86,623.63,2627.95
poupança,1306.58,5771.11,653.94,2362.34
salário,1407.26,6658.15,578.29,2861.86


### Contagem de status por tipo de transação

In [10]:
df.groupby(['transaction_type', 'status']).size().unstack(fill_value=0)

status,aprovada,estornada,recusada
transaction_type,,,
PIX,173,5,7
crédito,154,6,10
débito,266,6,15
transferência,141,8,9


## Funções Agregadas

Uma **função agregada** recebe um conjunto de valores e retorna um único valor que representa o grupo. São a base de qualquer sumarização de dados.

### `count` — contagem de valores não-nulos

Conta quantos valores existem na Series, **excluindo nulos**. Útil para detectar ausência de dados antes de qualquer análise.

In [11]:
df['amount'].count()

794

### `size` — contagem total de linhas

Conta **todas** as linhas do grupo, incluindo nulos. Quando chamado diretamente em uma Series, equivale a `len()`. A diferença com `count` aparece claramente no `groupby`:

In [12]:
df['amount'].size

800

In [13]:
print(df.groupby('transaction_type')['amount'].size())
print()
print(df.groupby('transaction_type')['amount'].count())

transaction_type
PIX              185
crédito          170
débito           287
transferência    158
Name: amount, dtype: int64

transaction_type
PIX              182
crédito          169
débito           285
transferência    158
Name: amount, dtype: int64


### `sum` — soma

Soma todos os valores não-nulos. Para colunas financeiras, retorna o volume total movimentado.

In [14]:
df['amount'].sum().round(2)

1890445.3

In [15]:
df.groupby('transaction_type')['amount'].sum()

transaction_type
PIX               237380.45
crédito          1067233.84
débito            178420.38
transferência     407410.63
Name: amount, dtype: float64


### `mean` — média aritmética

Soma dividida pela contagem de valores não-nulos. Representa o valor típico do grupo, mas é sensível a outliers.

In [16]:
df['amount'].mean().round(2)

2380.91

In [18]:
df.groupby('transaction_type')['amount'].mean().round(2)

transaction_type
PIX              1304.29
crédito          6314.99
débito            626.04
transferência    2578.55
Name: amount, dtype: float64

### `median` — mediana

Valor central quando os dados estão ordenados. Mais robusta que a média quando há outliers.

In [19]:
df['amount'].median()

1170.15

In [20]:
df.groupby('transaction_type')['amount'].median()

transaction_type
PIX              1357.055
crédito          6106.980
débito            673.810
transferência    2472.120
Name: amount, dtype: float64

### `std` — desvio padrão

Mede a dispersão dos valores em torno da média. Valores altos indicam grande variabilidade nas transações do grupo.

In [21]:
df['amount'].std()

2748.3974797163596

In [22]:
df.groupby('transaction_type')['amount'].std()

transaction_type
PIX               696.168409
crédito          3325.323466
débito            343.733050
transferência    1383.860319
Name: amount, dtype: float64

### `var` — variância

O quadrado do desvio padrão. Mede dispersão na mesma lógica, mas na unidade ao quadrado — o que torna a interpretação direta menos intuitiva. É mais usada internamente em cálculos estatísticos do que como métrica de negócio.

In [23]:
df['amount'].var()

7553688.706511238

In [24]:
df.groupby('transaction_type')['amount'].var()

transaction_type
PIX              4.846505e+05
crédito          1.105778e+07
débito           1.181524e+05
transferência    1.915069e+06
Name: amount, dtype: float64

### `min` e `max` — mínimo e máximo

Extremos absolutos do grupo. Úteis para validar dados (um `min` negativo em `amount` indicaria erro de entrada) e para entender a amplitude dos valores.

In [25]:
df.groupby('transaction_type')['amount'].agg(['min', 'max'])

,min,max
transaction_type,,
PIX,49.96,2496.95
crédito,821.78,11917.65
débito,24.66,1193.57
transferência,242.08,4995.52


### `first` e `last` — primeiro e último valor

Retornam o primeiro ou último valor do grupo na ordem em que os dados estão. Frequentemente usados com datas para identificar início e fim de atividade de um grupo.

In [27]:
df.groupby('state')['date'].agg(['min', 'max'])

,min,max
state,,
BA,2023-01-03,2024-06-11
GO,2023-01-09,2024-06-21
MG,2023-01-05,2024-06-28
PR,2023-01-13,2024-06-28
RJ,2023-01-02,2024-06-30
RS,2023-01-08,2024-06-20
SP,2023-01-04,2024-06-30


In [28]:
df.groupby('state')['date'].agg(['first', 'last'])

,first,last
state,,
BA,2023-01-03,2024-06-11
GO,2023-01-09,2024-06-21
MG,2023-01-05,2024-06-28
PR,2023-01-13,2024-06-28
RJ,2023-01-02,2024-06-30
RS,2023-01-08,2024-06-20
SP,2023-01-04,2024-06-30


### `nunique` — quantidade de valores distintos

Conta quantos valores únicos existem no grupo. Essencial para perguntas como "quantos clientes distintos fizeram PIX?".

In [29]:
df.groupby('transaction_type')['client_id'].nunique()

transaction_type
PIX              136
crédito          134
débito           182
transferência    125
Name: client_id, dtype: int64

### `idxmin` e `idxmax` — índice do mínimo e do máximo

Retornam o **índice da linha** onde o valor mínimo ou máximo ocorre. Útil para localizar um registro específico.

In [30]:
idx_max = df['amount'].idxmax()
print(df.loc[idx_max], ['transaction_id', 'client_id', 'transaction_type'])

transaction_id          100485
client_id                 5228
age_group                  56+
state                       GO
account_type          corrente
transaction_type       crédito
category             freelance
amount                11917.65
balance_after         13422.02
date                2023-07-27
status                recusada
Name: 287, dtype: object ['transaction_id', 'client_id', 'transaction_type']
